# 05m — Treino do Random Forest multiclasse (ajuste, validação, refit e custo computacional)

Este notebook treina o **RF multiclasse** (as 7 classes: `BENIGN` e 6 tipos de ataque) com a configuração inicial combinada, **mede o desempenho na validação** e **mede o custo computacional**. A busca de hiperparâmetros, o XAI e o SVM ficam para depois, e o **teste** é avaliado no notebook **06m**, uma única vez.

## Protocolo

| Fase | Dados | O que acontece |
|---|---|---|
| **1. Ajuste** | `prep_vw_bal_multi_treino` (239.671 linhas: até 40 mil por classe) | O RF é treinado, com `class_weight="balanced"` |
| **Validação** | `prep_vw_bal_multi_validacao` (59.918 linhas: até 10 mil por classe) | Métricas do modelo da fase 1, **dispersão em partições** e validação cruzada no ajuste |
| **2. Refit** | `prep_vw_bal_multi_refit` (criada aqui: **ajuste + validação = 299.589 linhas**, todas reais) | O modelo **final** é refeito com tudo. É ele que o notebook 06m avalia |
| **Teste** | notebook 06m | Usado uma vez |

**Como o refit é montado:** todas as linhas selecionadas pelo notebook 04m (as do ajuste e as da validação). A seleção fica gravada em `prep_bal_multi_refit_sel`, para o **SVM usar exatamente o mesmo conjunto** depois. Depois do refit, a validação **não mede mais nada** (as linhas dela entraram no treino): as métricas de qualidade vêm do modelo da fase 1, e o modelo final é avaliado só no teste.

**Pesos de classe.** O modelo usa `class_weight="balanced"`: cada classe pesa `linhas do treino ÷ (nº de classes × linhas da classe)`. No ajuste, isso dá ≈ 0,86 para as classes de 40 mil linhas, ≈ 1,42 para `LDAP` e ≈ 2,20 para `NetBIOS`. No refit, os pesos são recalculados com as linhas do refit. A previsão é a **classe de maior probabilidade** (não há limiar a escolher).

## Como ler as métricas

| Métrica | O que mede | Depende da proporção das classes? |
|---|---|---|
| **Recall por classe** | Fração das linhas da classe que o modelo acerta | Não |
| **Acurácia balanceada** | Média dos recalls das classes (macro) | Não |
| **Precisão e F1 por classe**, **F1 macro**, **F1 ponderado** | Dos que o modelo chamou de classe X, quantos eram X; harmônica com o recall | **Sim** |
| **Acurácia** | Fração de acertos | **Sim** |
| **MCC multiclasse** | Correlação entre previsão e verdade (−1 a 1); robusta a classes desiguais | **Sim** |
| **ROC-AUC um-contra-resto (macro)** | Para cada classe, quão bem ela é ordenada acima do resto; média das classes | Pouco (muda a mistura do "resto") |

**Duas colunas de resultado.** A validação tem as classes **equilibradas** (10 mil cada, menos `LDAP` e `NetBIOS`). A coluna **"proporção do treino"** usa o peso `peso` da view de validação, que devolve as proporções das 7 classes no treino (`UDP` ≈ 66%, `MSSQL` ≈ 13%, e assim por diante). **As decisões de modelo usam a coluna sem pesos.** A ponderada serve para relatar precisão, F1 e MCC numa mistura realista do treino. **O teste tem outra mistura** (o `Syn` é 68%): lá, os números globais refletem quase só o `Syn`, por isso olhamos também o recall por classe.

## Custo computacional (medido durante o treino)

| Medida | Como |
|---|---|
| **Tempo de treino** | Relógio (`perf_counter`) e **tempo de CPU**. O refit é repetido `REPETICOES_TEMPO` vezes (mediana, mínimo e máximo) |
| **Memória** | **Pico de RAM** do processo durante o treino (a cada 50 ms, com `psutil`), tamanho do modelo em disco e número de nós das árvores |
| **Inferência** | **Latência** de 1 fluxo por vez (p50, p95, p99) e **vazão** em lote (fluxos/s), com `n_jobs=1` e com todos os núcleos, **com e sem o pré-processamento**. A leitura do banco fica de fora |

## O que é salvo (pasta `artefatos_modelos/rf_multiclasse/rfm_<data>`)

`modelo_ajuste.joblib`, `modelo_final_refit.joblib`, `config_treino.json` (hiperparâmetros, **classes e pesos**, sementes, versões, dados, software e hardware), `preprocessamento.json` (**nomes e ordem dos atributos**, medianas, indicadores, colunas retiradas), `classes.json`, `metricas_validacao.json/csv`, `metricas_por_classe_validacao.csv`, `matriz_confusao_validacao.csv`, `dispersao_particoes_validacao.csv`, `cv_ajuste.csv`, `cv_recall_por_classe.csv`, `custo_computacional.json/csv`, `refit_ids.csv` e uma linha nova em `registro_experimentos.csv`.

## Antes de rodar
Os notebooks 02, 03m e 04m precisam ter terminado. Coloque este arquivo na **mesma pasta** deles (onde está o `.env`). Bibliotecas: `scikit-learn`, `joblib` e `psutil`.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


## 1. Parâmetros e o que existe
`HIPERPARAMETROS` é a configuração inicial (a **mesma do binário**, mais `class_weight="balanced"`). A célula confere que as views e tabelas dos notebooks 03m e 04m existem e mostra as classes e o plano de linhas.

In [6]:
# ---------- 1.1 Parâmetros ----------
HIPERPARAMETROS = globals().get("HIPERPARAMETROS", dict(n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1))
SEMENTE = globals().get("SEMENTE", 42)                       # partições, validação cruzada, refit e amostras de latência
K_PARTICOES_VAL = globals().get("K_PARTICOES_VAL", 5)
K_CV = globals().get("K_CV", 5)
REPETICOES_TEMPO = globals().get("REPETICOES_TEMPO", 3)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ARTEFATOS_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_multiclasse"
PASTA = ARTEFATOS_BASE / f"rfm_{RUN_ID}"; PASTA.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT = "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao", "prep_vw_bal_multi_refit"
SEL_TAB, DADOS_TAB, TAB_REFIT = "prep_bal_multi_selecao", "prep_bal_multi_dados", "prep_bal_multi_refit_sel"
AUD_TAB, TAB_CLASSES = "prep_views_rf_multi", "prep_classes_rf_multi"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, SEL_TAB, DADOS_TAB, AUD_TAB, TAB_CLASSES) if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03m e 04m: faltam {faltam}"
assert HIPERPARAMETROS.get("class_weight") == "balanced", "a configuração do multiclasse usa class_weight='balanced' (decisão do protocolo)"

versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num(f"SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM {AUD_TAB} ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
cl = consultar_num(f"SELECT codigo, classe, linhas_treino, linhas_teste_sem_sobrep FROM {TAB_CLASSES} ORDER BY codigo")
CLASSES = cl.classe.tolist(); K = len(CLASSES); COD = {c: int(k) for c, k in zip(cl.classe, cl.codigo)}; N_TREINO = {c: int(n) for c, n in zip(cl.classe, cl.linhas_treino)}
assert [COD[c] for c in CLASSES] == list(range(K)), "os códigos das classes precisam ser 0..K-1"
cont = consultar_num(f"SELECT papel, y, count() AS n FROM {SEL_TAB} GROUP BY papel, y"); cont = {(r.papel, int(r.y)): int(r.n) for r in cont.itertuples()}
N_AJ = {c: cont[("treino", COD[c])] for c in CLASSES}; N_VA = {c: cont[("validacao", COD[c])] for c in CLASSES}
print(f"Pasta dos artefatos: {PASTA}")
print(f"Seleção do notebook 04m (versão {versao_sel}) | views do notebook 03m (versão {versao_views})")
print(f"{K} classes: {CLASSES}")
print(pd.DataFrame({"no treino": N_TREINO, "ajuste": N_AJ, "validação": N_VA}).T.to_string())
print("Hiperparâmetros:", HIPERPARAMETROS)

Pasta dos artefatos: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse\rfm_20261101_210334
Seleção do notebook 04m (versão 20261027_173009) | views do notebook 03m (versão 20261026_010742)
7 classes: ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag']
           BENIGN   LDAP   MSSQL  NetBIOS     Syn      UDP  UDPLag
no treino   50768  30099  207003    19490  155515  1075337   89066
ajuste      40000  24079   40000    15592   40000    40000   40000
validação   10000   6020   10000     3898   10000    10000   10000
Hiperparâmetros: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}


## 2. Conjunto do refit
O refit usa **todas as linhas selecionadas pelo notebook 04m** (ajuste + validação = até 50 mil reais por classe). Aqui ele é gravado em `prep_bal_multi_refit_sel` (auditoria) e exposto na view `prep_vw_bal_multi_refit`, que lê a tabela de dados já pronta. Os pesos `balanced` do refit são calculados pelo próprio RF, no momento do treino, a partir das linhas do refit.

In [7]:
# ---------- 2.1 Montagem e gravação do refit ----------
sel = consultar(f"SELECT toString(id_linha) AS id, papel, y, tipo FROM {SEL_TAB}"); sel["y"] = sel["y"].astype(int)
refit_sel = sel.assign(origem=np.where(sel.papel == "treino", "ajuste", "validacao"))
assert refit_sel.id.is_unique and refit_sel.id.map(str.isdigit).all()
assert len(refit_sel) == sum(N_AJ.values()) + sum(N_VA.values())

executar(f"DROP TABLE IF EXISTS `{TAB_REFIT}`")
executar(f"CREATE TABLE {TAB_REFIT} (id_linha UInt64, origem String, tipo String, versao String) ENGINE = MergeTree ORDER BY id_linha")
for i in range(0, len(refit_sel), 20_000):
    p = refit_sel.iloc[i:i + 20_000]
    executar_robusto(f"INSERT INTO {TAB_REFIT} VALUES " + ", ".join(f"({r.id}, {lit(r.origem)}, {lit(r.tipo)}, {lit(RUN_ID)})" for r in p.itertuples()))
cols_modelo_sql = ", ".join(q(c) for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y"))
executar(f"DROP VIEW IF EXISTS `{V_REFIT}`")
executar(f"CREATE VIEW {V_REFIT} AS SELECT id_linha, y, {cols_modelo_sql} FROM {DADOS_TAB} WHERE id_linha IN (SELECT id_linha FROM {TAB_REFIT})")
refit_sel[["id", "origem", "tipo", "y"]].rename(columns={"id": "id_linha"}).to_csv(PASTA / "refit_ids.csv", index=False)
print(f"✔ {TAB_REFIT}: {n_linhas(TAB_REFIT):,} linhas | view {V_REFIT}")
print(refit_sel.groupby(["tipo", "origem"]).size().unstack(fill_value=0).reindex(CLASSES).to_string())

✔ prep_bal_multi_refit_sel: 299,589 linhas | view prep_vw_bal_multi_refit
origem   ajuste  validacao
tipo                      
BENIGN    40000      10000
LDAP      24079       6020
MSSQL     40000      10000
NetBIOS   15592       3898
Syn       40000      10000
UDP       40000      10000
UDPLag    40000      10000


## 3. Pré-processamento e leitura dos dados
Monta o `PreprocessadorRF` com a auditoria do notebook 03m e **confere que ele reproduz a matriz das views**: transformando os valores brutos (com os vazios de volta), ele devolve exatamente o que a view entrega. Assim, o modelo gravado aceita **dados crus** na hora de usar. As views têm poucos milhares de linhas por classe, então são lidas inteiras.

In [8]:
# ---------- 3.1 Pré-processamento e dados ----------
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views)
assert prep.colunas_modelo == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
print(f"Atributos do modelo: {len(prep.colunas_modelo)} ({len(prep.features)} + {len(prep.indicadores)} indicadores) | retiradas na ablação: {RETIRADAS}")

aj, va, rf_df = ler_view(V_AJUSTE), ler_view(V_VALID), ler_view(V_REFIT)
CM = prep.colunas_modelo
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int)
X_va, y_va, w_va = va[CM].to_numpy(float), va["y"].to_numpy(int), va["peso"].to_numpy(float)
X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va), ("refit", X_rf)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"

X_va_raw = X_va[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_va_raw[X_va[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_va_raw), X_va, rtol=1e-12, atol=1e-12), "o PreprocessadorRF não reproduz a matriz da view"
print(f"✔ PreprocessadorRF reproduz a matriz das views ({len(X_va):,} linhas com {int(np.isnan(X_va_raw).any(axis=1).sum()):,} fluxos com algum valor ausente)")
conta = lambda y: {CLASSES[k]: int((y == k).sum()) for k in range(K)}
print(f"Ajuste: {len(X_aj):,} linhas {conta(y_aj)}\nValidação: {len(X_va):,} linhas {conta(y_va)}\nRefit: {len(X_rf):,} linhas {conta(y_rf)}")

Atributos do modelo: 44 (42 + 2 indicadores) | retiradas na ablação: ['bwd_packets_s', 'inbound', 'total_backward_packets']
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "id_linha",\n\t\t\t"type": "String"\n\t\t},\n\t\t{\n\t\t\t"name": "y",\n\t\t\t"); nova tentativa em 5s...
✔ PreprocessadorRF reproduz a matriz das views (59,918 linhas com 1,207 fluxos com algum valor ausente)
Ajuste: 239,671 linhas {'BENIGN': 40000, 'LDAP': 24079, 'MSSQL': 40000, 'NetBIOS': 15592, 'Syn': 40000, 'UDP': 40000, 'UDPLag': 40000}
Validação: 59,918 linhas {'BENIGN': 10000, 'LDAP': 6020, 'MSSQL': 10000, 'NetBIOS': 3898, 'Syn': 10000, 'UDP': 10000, 'UDPLag': 10000}
Refit: 299,589 linhas {'BENIGN': 50000, 'LDAP': 30099, 'MSSQL': 50000, 'NetBIOS': 19490, 'Syn': 50000, 'UDP': 50000, 'UDPLag': 50000}


## 4. Fase 1 — treino no ajuste e avaliação na validação
Treina o RF com o ajuste (`class_weight="balanced"`) e mede, **na validação** (linhas que ele nunca viu):
1. métricas **sem pesos** e na **proporção do treino**, recall e precisão **por classe** e a **matriz de confusão** (4.2);
2. a **dispersão entre 5 partições** da validação (4.3), para saber quanto os números balançam;
3. a **validação cruzada no ajuste** (4.4), com 5 dobras estratificadas por classe, com o mesmo RF e os pesos recalculados em cada dobra.

Onde as classes se **confundem** de verdade (por exemplo, `MSSQL` × `NetBIOS` ou `UDP` × `UDPLag`) aparece na matriz de confusão e na lista das confusões mais frequentes. Lembre que parte dessas confusões vem de linhas com **as mesmas features e rótulos diferentes** (os conflitos de rótulo do notebook 02): nesses casos nenhum modelo acerta os dois lados.

In [9]:
# ---------- 4.1 Treino no ajuste ----------
with Medidor() as med_aj:
    rf = RandomForestClassifier(**HIPERPARAMETROS).fit(X_aj, y_aj)
rf.colunas_modelo_ = list(CM)                                    # a ordem dos atributos viaja dentro do arquivo do modelo
assert list(rf.classes_) == list(range(K)), "o modelo não aprendeu todas as classes (ou a ordem delas mudou)"
fit_ajuste = {"relogio_s": med_aj.relogio, "cpu_s": med_aj.cpu, "pico_ram_mib": med_aj.pico_mib, "acrescimo_ram_mib": med_aj.delta_mib}
print(f"✔ RF treinado no ajuste: {med_aj.relogio:.1f} s de relógio, {med_aj.cpu:.1f} s de CPU, pico de RAM {med_aj.pico_mib:,.0f} MiB (+{med_aj.delta_mib:,.0f} MiB sobre o início)")
PESO_CLASSE_AJ = {CLASSES[k]: float(len(y_aj) / (K * (y_aj == k).sum())) for k in range(K)}
print("Pesos de classe do ajuste (balanced):", {c: round(v, 3) for c, v in PESO_CLASSE_AJ.items()})

# ---------- 4.2 Métricas na validação ----------
proba_va = rf.predict_proba(X_va)
m_sem, arr_sem = metricas_multi(y_va, proba_va, None); m_prop, arr_prop = metricas_multi(y_va, proba_va, w_va)
DEPENDE = {k: "sim" for k in ("acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")}; DEPENDE["roc_auc_ovr_macro"] = "pouco"
ordem = ["roc_auc_ovr_macro", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc"]
tab_val = pd.DataFrame({"sem pesos": {k: m_sem[k] for k in ordem}, "proporção do treino": {k: m_prop[k] for k in ordem}}); tab_val["depende da proporção?"] = [DEPENDE.get(k, "não") for k in tab_val.index]
print("\n== Métricas na validação (previsão = classe de maior probabilidade) ==  (a coluna 'proporção do treino' usa o peso de cada linha da validação)")
display(tab_val.round(5))
tab_cls = pd.DataFrame({"suporte (validação)": arr_sem["suporte"].astype(int), "recall": arr_sem["recall"], "precisão (sem pesos)": arr_sem["precisao"], "F1 (sem pesos)": arr_sem["f1"], "precisão (prop. treino)": arr_prop["precisao"], "F1 (prop. treino)": arr_prop["f1"],
                        "ROC-AUC um-contra-resto": arr_sem["auc"], "peso de classe no ajuste": [PESO_CLASSE_AJ[c] for c in CLASSES]}, index=CLASSES)
print("\n== Por classe (recall e ROC-AUC não dependem da proporção) =="); display(tab_cls.round(4))
cm_val = pd.DataFrame(arr_sem["cm"].astype(int), index=[f"real {c}" for c in CLASSES], columns=[f"prev. {c}" for c in CLASSES])
print("\n== Matriz de confusão na validação (linhas = classe real; colunas = classe prevista) =="); display(cm_val)
cm_rec = pd.DataFrame(arr_sem["cm"] / arr_sem["cm"].sum(axis=1, keepdims=True), index=cm_val.index, columns=cm_val.columns)
print("== Mesma matriz, em fração da classe real (a diagonal é o recall) =="); display(cm_rec.round(4))
fora = [(CLASSES[i], CLASSES[j], int(arr_sem["cm"][i, j]), arr_sem["cm"][i, j] / arr_sem["cm"][i].sum()) for i in range(K) for j in range(K) if i != j and arr_sem["cm"][i, j] > 0]
fora = sorted(fora, key=lambda x: -x[2])[:8]
print("Confusões mais frequentes (real → previsto): " + " | ".join(f"{a}→{b}: {n:,} ({100 * f:.2f}% de {a})" for a, b, n, f in fora))

# ---------- 4.3 Dispersão entre partições da validação ----------
linhas_part = []
for k, (_, idx) in enumerate(StratifiedKFold(K_PARTICOES_VAL, shuffle=True, random_state=SEMENTE).split(X_va, y_va), 1):
    e1, _a = metricas_multi(y_va[idx], proba_va[idx], None); e2, _b = metricas_multi(y_va[idx], proba_va[idx], w_va[idx])
    linhas_part.append({"particao": k, "n": len(idx), "roc_auc_ovr_macro": e1["roc_auc_ovr_macro"], "acuracia_balanceada": e1["acuracia_balanceada"], "acuracia": e1["acuracia"], "f1_macro": e1["f1_macro"], "mcc": e1["mcc"], "f1_macro_prop": e2["f1_macro"], "mcc_prop": e2["mcc"]})
tab_part = pd.DataFrame(linhas_part); resumo_part = tab_part.drop(columns=["particao", "n"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Dispersão entre {K_PARTICOES_VAL} partições da validação =="); display(tab_part.round(5)); display(resumo_part.round(5))

# ---------- 4.4 Validação cruzada no ajuste ----------
linhas_cv, linhas_cv_cls = [], []
if K_CV and K_CV > 1:
    for k, (itr, ite) in enumerate(StratifiedKFold(K_CV, shuffle=True, random_state=SEMENTE).split(X_aj, y_aj), 1):
        t0 = time.perf_counter(); mk = RandomForestClassifier(**HIPERPARAMETROS).fit(X_aj[itr], y_aj[itr]); tf = time.perf_counter() - t0
        e, a = metricas_multi(y_aj[ite], mk.predict_proba(X_aj[ite]), None)
        linhas_cv.append({"dobra": k, "n_treino": len(itr), "n_avaliacao": len(ite), "tempo_treino_s": tf, **{c: e[c] for c in ("roc_auc_ovr_macro", "acuracia_balanceada", "acuracia", "f1_macro", "mcc")}})
        linhas_cv_cls.append({"dobra": k, **{f"recall_{CLASSES[j]}": a["recall"][j] for j in range(K)}})
    tab_cv = pd.DataFrame(linhas_cv); resumo_cv = tab_cv.drop(columns=["dobra", "n_treino", "n_avaliacao"]).agg(["mean", "std", "min", "max"]).T; tab_cv_cls = pd.DataFrame(linhas_cv_cls)
    print(f"\n== Validação cruzada no ajuste ({K_CV} dobras estratificadas por classe) =="); display(tab_cv.round(5)); display(resumo_cv.round(5))
    print("Recall por classe em cada dobra:"); display(tab_cv_cls.round(4))
else:
    tab_cv, resumo_cv, tab_cv_cls = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(); print("\n(validação cruzada no ajuste desligada: K_CV = 0)")

✔ RF treinado no ajuste: 41.3 s de relógio, 258.7 s de CPU, pico de RAM 1,401 MiB (+542 MiB sobre o início)
Pesos de classe do ajuste (balanced): {'BENIGN': 0.856, 'LDAP': 1.422, 'MSSQL': 0.856, 'NetBIOS': 2.196, 'Syn': 0.856, 'UDP': 0.856, 'UDPLag': 0.856}

== Métricas na validação (previsão = classe de maior probabilidade) ==  (a coluna 'proporção do treino' usa o peso de cada linha da validação)


,sem pesos,proporção do treino,depende da proporção?
roc_auc_ovr_macro,0.99020,0.98878,pouco
acuracia_balanceada,0.90255,0.90255,não
acuracia,0.91265,0.91390,sim
precisao_macro,0.89916,0.80403,sim
f1_macro,0.90069,0.83519,sim
f1_ponderado,0.91271,0.92490,sim
mcc,0.89689,0.85201,sim



== Por classe (recall e ROC-AUC não dependem da proporção) ==


,suporte (validação),recall,precisão (sem pesos),F1 (sem pesos),precisão (prop. treino),F1 (prop. treino),ROC-AUC um-contra-resto,peso de classe no ajuste
BENIGN,10000,0.9990,0.9987,0.9989,0.9974,0.9982,1.0000,0.8560
LDAP,6020,0.9646,0.9595,0.9621,0.8740,0.9171,0.9983,1.4219
MSSQL,10000,0.8804,0.8986,0.8894,0.9628,0.9198,0.9863,0.8560
NetBIOS,3898,0.7630,0.7234,0.7427,0.3804,0.5077,0.9792,2.1959
Syn,10000,0.9550,0.9502,0.9526,0.9704,0.9626,0.9965,0.8560
UDP,10000,0.9180,0.8926,0.9051,0.9900,0.9526,0.9914,0.8560
UDPLag,10000,0.8379,0.8710,0.8541,0.4533,0.5884,0.9798,0.8560



== Matriz de confusão na validação (linhas = classe real; colunas = classe prevista) ==


,prev. BENIGN,prev. LDAP,prev. MSSQL,prev. NetBIOS,prev. Syn,prev. UDP,prev. UDPLag
real BENIGN,9990,3,0,3,2,1,1
real LDAP,1,5807,135,71,2,0,4
real MSSQL,1,171,8804,999,0,18,7
real NetBIOS,4,56,831,2974,2,11,20
real Syn,4,3,0,5,9550,0,438
real UDP,0,2,20,26,1,9180,771
real UDPLag,3,10,7,33,494,1074,8379


... (6 outputs left out)

,dobra,n_treino,n_avaliacao,tempo_treino_s,roc_auc_ovr_macro,acuracia_balanceada,acuracia,f1_macro,mcc
0,1,191736,47935,34.08019,0.99063,0.90456,0.91466,0.90247,0.89929
1,2,191737,47934,45.79476,0.99080,0.90161,0.91194,0.89960,0.89609
2,3,191737,47934,45.45891,0.99050,0.90482,0.91474,0.90313,0.89937
3,4,191737,47934,44.59409,0.98997,0.90326,0.91307,0.90129,0.89746
4,5,191737,47934,40.83979,0.98995,0.90254,0.91311,0.90091,0.89743


,mean,std,min,max
tempo_treino_s,42.15355,4.92557,34.08019,45.79476
roc_auc_ovr_macro,0.99037,0.00039,0.98995,0.99080
acuracia_balanceada,0.90336,0.00135,0.90161,0.90482
acuracia,0.91350,0.00119,0.91194,0.91474
f1_macro,0.90148,0.00138,0.89960,0.90313
mcc,0.89793,0.00140,0.89609,0.89937


Recall por classe em cada dobra:


,dobra,recall_BENIGN,recall_LDAP,recall_MSSQL,recall_NetBIOS,recall_Syn,recall_UDP,recall_UDPLag
0,1,0.9992,0.9637,0.8841,0.7669,0.9585,0.9226,0.8369
1,2,0.9985,0.9657,0.8798,0.7583,0.9552,0.9215,0.8322
2,3,0.9995,0.9637,0.8824,0.7691,0.9555,0.9260,0.8376
3,4,0.9996,0.9682,0.8820,0.7646,0.9585,0.9265,0.8234
4,5,0.9995,0.9643,0.8802,0.7585,0.9586,0.9168,0.8399


## 5. Custo computacional do modelo da fase 1
Tamanho do arquivo, número de árvores e de nós, **latência** de 1 fluxo por vez (com `n_jobs=1` e com todos os núcleos) e **vazão** em lote, **com e sem o pré-processamento**. As previsões são medidas na validação, **só para velocidade**.

In [10]:
# ---------- 5.1 Custo do modelo da fase 1 ----------
tam_aj = tamanho_e_nos(rf, PASTA / "modelo_ajuste.joblib", COMPRESSAO_MODELO)
inf_aj = medir_inferencia(rf, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"Modelo do ajuste: {tam_aj['arquivo_mib']:.1f} MiB em disco | {tam_aj['n_arvores']} árvores | {tam_aj['nos_total']:,} nós | profundidade média {tam_aj['profundidade_media']:.1f} (máx {tam_aj['profundidade_max']})")
display(pd.DataFrame(inf_aj).set_index("n_jobs").T.round(3))

Modelo do ajuste: 54.7 MiB em disco | 100 árvores | 3,483,364 nós | profundidade média 61.7 (máx 74)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,13.487,51.344
latencia_modelo_p95_ms,20.255,68.978
latencia_modelo_p99_ms,21.970,79.624
latencia_fim_a_fim_p50_ms,13.711,51.503
latencia_fim_a_fim_p95_ms,20.668,69.200
latencia_fim_a_fim_p99_ms,22.653,80.248
vazao_modelo_fluxos_s,19588.817,71073.657
vazao_fim_a_fim_fluxos_s,22540.263,62396.534
n_lote,59918.000,59918.000


## 6. Fase 2 — refit (modelo final)
Refaz o RF com **todas as linhas do refit** (ajuste + validação, `class_weight="balanced"` recalculado) `REPETICOES_TEMPO` vezes, só para **medir o tempo de forma confiável** (mediana, mínimo e máximo). O notebook confere que as repetições dão o **mesmo modelo** (a semente fixa o resultado). O primeiro modelo é o **modelo final**, gravado para o notebook 06m.

In [11]:
# ---------- 6.1 Refit ----------
reps, modelo_final, p_ref = [], None, None
for k in range(1, REPETICOES_TEMPO + 1):
    with Medidor() as m:
        mdl = RandomForestClassifier(**HIPERPARAMETROS).fit(X_rf, y_rf)
    reps.append({"repeticao": k, "relogio_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib})
    if modelo_final is None:
        modelo_final = mdl; modelo_final.colunas_modelo_ = list(CM); p_ref = mdl.predict_proba(X_va[:5000])
    else:
        assert np.allclose(mdl.predict_proba(X_va[:5000]), p_ref), "as repetições do refit deram modelos diferentes (a semente não está fixando o resultado)"
    del mdl
assert list(modelo_final.classes_) == list(range(K))
tab_reps = pd.DataFrame(reps); display(tab_reps.round(3))
fit_refit = {"repeticoes": REPETICOES_TEMPO, "relogio_mediana_s": float(tab_reps.relogio_s.median()), "relogio_min_s": float(tab_reps.relogio_s.min()), "relogio_max_s": float(tab_reps.relogio_s.max()),
             "cpu_mediana_s": float(tab_reps.cpu_s.median()), "pico_ram_max_mib": float(tab_reps.pico_ram_mib.max()), "acrescimo_ram_max_mib": float(tab_reps.acrescimo_ram_mib.max())}
PESO_CLASSE_RF = {CLASSES[k]: float(len(y_rf) / (K * (y_rf == k).sum())) for k in range(K)}
tam_fin = tamanho_e_nos(modelo_final, PASTA / "modelo_final_refit.joblib", COMPRESSAO_MODELO)
inf_fin = medir_inferencia(modelo_final, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE)
print(f"\n✔ Refit: {fit_refit['relogio_mediana_s']:.1f} s (mediana de {REPETICOES_TEMPO}; mín {fit_refit['relogio_min_s']:.1f}, máx {fit_refit['relogio_max_s']:.1f}), CPU {fit_refit['cpu_mediana_s']:.1f} s, pico de RAM {fit_refit['pico_ram_max_mib']:,.0f} MiB")
print(f"Pesos de classe do refit (balanced): { {c: round(v, 3) for c, v in PESO_CLASSE_RF.items()} }")
print(f"Modelo final: {tam_fin['arquivo_mib']:.1f} MiB em disco | {tam_fin['nos_total']:,} nós | profundidade média {tam_fin['profundidade_media']:.1f} (máx {tam_fin['profundidade_max']})")
print("(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)")
display(pd.DataFrame(inf_fin).set_index("n_jobs").T.round(3))

,repeticao,relogio_s,cpu_s,pico_ram_mib,acrescimo_ram_mib
0,1,69.959,372.250,1502.625,562.266
1,2,74.246,401.234,1472.250,236.781
2,3,55.114,355.406,1337.328,276.613



✔ Refit: 70.0 s (mediana de 3; mín 55.1, máx 74.2), CPU 372.2 s, pico de RAM 1,503 MiB
Pesos de classe do refit (balanced): {'BENIGN': 0.856, 'LDAP': 1.422, 'MSSQL': 0.856, 'NetBIOS': 2.196, 'Syn': 0.856, 'UDP': 0.856, 'UDPLag': 0.856}
Modelo final: 67.1 MiB em disco | 4,288,578 nós | profundidade média 65.1 (máx 96)
(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)


n_jobs,1,-1
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,9.348,48.581
latencia_modelo_p95_ms,17.167,59.416
latencia_modelo_p99_ms,20.045,71.896
latencia_fim_a_fim_p50_ms,9.563,48.687
latencia_fim_a_fim_p95_ms,17.575,59.722
latencia_fim_a_fim_p99_ms,20.351,72.401
vazao_modelo_fluxos_s,33700.601,66370.633
vazao_fim_a_fim_fluxos_s,33993.255,60683.885
n_lote,59918.000,59918.000


## 7. Gravação dos registros
Grava na pasta da execução os dois modelos, a configuração (com as **classes e os pesos de classe**), o pré-processamento (nomes e ordem dos atributos, medianas), as métricas da validação (incluindo por classe e a matriz de confusão), a dispersão entre partições, a validação cruzada e o custo. Acrescenta uma linha a `registro_experimentos.csv`, base da futura busca de hiperparâmetros.

In [12]:
# ---------- 7.1 Gravação ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
config = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "modelo": "RandomForestClassifier (multiclasse)", "hiperparametros": HIPERPARAMETROS, "semente": SEMENTE,
          "previsao": "classe de maior probabilidade (argmax)", "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views,
          "classes": {"nomes": CLASSES, "codigos": COD}, "pesos_classe_balanced_ajuste": PESO_CLASSE_AJ, "pesos_classe_balanced_refit": PESO_CLASSE_RF,
          "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "refit": len(X_rf), "linhas_treino_por_classe": N_TREINO, "ajuste_por_classe": N_AJ, "validacao_por_classe": N_VA, "usadas_por_classe": {c: N_AJ[c] + N_VA[c] for c in CLASSES}},
          "protocolo": {"k_particoes_validacao": K_PARTICOES_VAL, "k_cv_ajuste": K_CV, "repeticoes_tempo_refit": REPETICOES_TEMPO, "n_latencia": N_LATENCIA}, "software": sw, "hardware": hw}
salvar_json(config, PASTA / "config_treino.json"); salvar_json(prep.para_dict(), PASTA / "preprocessamento.json"); salvar_json({"classes": CLASSES, "codigos": COD}, PASTA / "classes.json")
met_val = {"sem_pesos": m_sem, "proporcao_treino": m_prop, "por_classe": tab_cls.to_dict(orient="index"), "matriz_confusao": cm_val.to_dict(orient="index"),
           "dispersao_particoes": {"k": K_PARTICOES_VAL, "resumo": resumo_part.to_dict(orient="index")}, "cv_ajuste": {"k": K_CV, "resumo": resumo_cv.to_dict(orient="index")}}
salvar_json(met_val, PASTA / "metricas_validacao.json")
tab_val.reset_index(names="metrica").to_csv(PASTA / "metricas_validacao.csv", index=False); tab_cls.reset_index(names="classe").to_csv(PASTA / "metricas_por_classe_validacao.csv", index=False)
cm_val.reset_index(names="classe_real").to_csv(PASTA / "matriz_confusao_validacao.csv", index=False)
tab_part.to_csv(PASTA / "dispersao_particoes_validacao.csv", index=False); tab_cv.to_csv(PASTA / "cv_ajuste.csv", index=False); tab_cv_cls.to_csv(PASTA / "cv_recall_por_classe.csv", index=False)
custo = {"fit_ajuste": fit_ajuste, "fit_refit": fit_refit, "fit_refit_repeticoes": reps, "modelo_ajuste": tam_aj, "modelo_final": tam_fin, "inferencia_modelo_ajuste": inf_aj, "inferencia_modelo_final": inf_fin}
salvar_json(custo, PASTA / "custo_computacional.json")
linhas_custo = [{"item": "fit_ajuste", **fit_ajuste}, {"item": "fit_refit", **{k: v for k, v in fit_refit.items()}}] + [{"item": f"inferencia_final_njobs{r['n_jobs']}", **r} for r in inf_fin] + [{"item": "modelo_final", **tam_fin}]
pd.DataFrame(linhas_custo).to_csv(PASTA / "custo_computacional.csv", index=False)

i1 = next(r for r in inf_fin if r["n_jobs"] == 1); iN = inf_fin[-1]
registro = {"run_id": RUN_ID, "data": config["data"], "modelo": "RandomForest multiclasse", "hiperparametros": json.dumps(_limpar(HIPERPARAMETROS)), "n_ajuste": len(X_aj), "n_validacao": len(X_va), "n_refit": len(X_rf), "n_classes": K,
            "versao_selecao": versao_sel, "versao_views": versao_views, "roc_auc_ovr_val": m_sem["roc_auc_ovr_macro"], "acuracia_balanceada_val": m_sem["acuracia_balanceada"], "f1_macro_val": m_sem["f1_macro"], "mcc_val": m_sem["mcc"],
            "f1_macro_prop_val": m_prop["f1_macro"], "mcc_prop_val": m_prop["mcc"], "acuracia_prop_val": m_prop["acuracia"],
            "particoes_f1_macro_media": resumo_part.loc["f1_macro", "mean"], "particoes_f1_macro_dp": resumo_part.loc["f1_macro", "std"],
            "cv_f1_macro_media": resumo_cv.loc["f1_macro", "mean"] if len(resumo_cv) else float("nan"), "cv_f1_macro_dp": resumo_cv.loc["f1_macro", "std"] if len(resumo_cv) else float("nan"),
            "cv_mcc_media": resumo_cv.loc["mcc", "mean"] if len(resumo_cv) else float("nan"), "cv_mcc_dp": resumo_cv.loc["mcc", "std"] if len(resumo_cv) else float("nan"),
            "fit_ajuste_relogio_s": fit_ajuste["relogio_s"], "fit_ajuste_cpu_s": fit_ajuste["cpu_s"], "fit_ajuste_pico_ram_mib": fit_ajuste["pico_ram_mib"],
            "fit_refit_relogio_mediana_s": fit_refit["relogio_mediana_s"], "modelo_final_mib": tam_fin["arquivo_mib"], "nos_total": tam_fin["nos_total"],
            "latencia_p50_ms_njobs1": i1["latencia_fim_a_fim_p50_ms"], "latencia_p99_ms_njobs1": i1["latencia_fim_a_fim_p99_ms"], "vazao_fluxos_s_njobs1": i1["vazao_fim_a_fim_fluxos_s"],
            "vazao_fluxos_s_njobs_orig": iN["vazao_fim_a_fim_fluxos_s"], "pasta": str(PASTA)}
caminho_reg = ARTEFATOS_BASE / "registro_experimentos.csv"
pd.DataFrame([registro]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"✔ Artefatos gravados em {PASTA}\n✔ Registro acrescentado a {caminho_reg} ({sum(1 for _ in open(caminho_reg, encoding='utf-8')) - 1} execução(ões) registrada(s))")
for f in sorted(PASTA.iterdir()): print(f"   {f.name:38s} {f.stat().st_size / 2**10:>10,.1f} KiB")

✔ Artefatos gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse\rfm_20261101_210334
✔ Registro acrescentado a C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse\registro_experimentos.csv (1 execução(ões) registrada(s))
   classes.json                                  0.2 KiB
   config_treino.json                            2.5 KiB
   custo_computacional.csv                       1.1 KiB
   custo_computacional.json                      3.2 KiB
   cv_ajuste.csv                                 0.7 KiB
   cv_recall_por_classe.csv                      0.5 KiB
   dispersao_particoes_validacao.csv             0.8 KiB
   matriz_confusao_validacao.csv                 0.3 KiB
   metricas_por_classe_validacao.csv             1.1 KiB
   metricas_validacao.csv                        0.4 KiB
   metricas_validacao.json                       6.5 KiB
   modelo_ajuste.joblib                     55,964.6 KiB
   modelo_final_refit.joblib

## 8. Conferências finais
Confere, de forma independente, que os arquivos gravados reproduzem o que está em memória, que o ajuste e o refit têm as linhas planejadas por classe, que o modelo aprendeu as 7 classes na ordem certa, que as probabilidades e as matrizes de confusão fecham, que os **pesos da validação** somam o tamanho de cada classe no treino e que as métricas que não dependem da proporção são iguais nas duas colunas.

In [13]:
# ---------- 8.1 Conferências ----------
from sklearn.utils.class_weight import compute_class_weight
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

m_ajuste_lido = joblib.load(PASTA / "modelo_ajuste.joblib"); m_final_lido = joblib.load(PASTA / "modelo_final_refit.joblib")
verifica("Modelo do ajuste gravado reproduz as probabilidades da validação", np.allclose(m_ajuste_lido.predict_proba(X_va), proba_va))
verifica("Modelo final gravado reproduz as previsões do modelo em memória", np.allclose(m_final_lido.predict_proba(X_va[:5000]), p_ref))
prep_lido = json.loads((PASTA / "preprocessamento.json").read_text(encoding="utf-8"))
verifica("Ordem dos atributos gravada = ordem usada no treino e dentro do modelo", prep_lido["colunas_modelo"] == CM == list(m_final_lido.colunas_modelo_) == list(m_ajuste_lido.colunas_modelo_))
verifica("Medianas gravadas = medianas da tabela de auditoria do notebook 03m", prep_lido["medianas_treino"] == MEDIANAS and set(prep_lido["colunas_retiradas_ablacao"]) == set(RETIRADAS))
verifica("Hiperparâmetros do modelo = configuração escolhida (inclui class_weight='balanced')", all(getattr(m_final_lido, k) == v for k, v in HIPERPARAMETROS.items()) and m_final_lido.class_weight == "balanced")
cls_lido = json.loads((PASTA / "classes.json").read_text(encoding="utf-8"))
verifica("O modelo aprendeu as classes 0..K-1, na ordem de classes.json", list(m_final_lido.classes_) == list(m_ajuste_lido.classes_) == list(range(K)) == [cls_lido["codigos"][c] for c in cls_lido["classes"]], f"({K} classes)")
verifica("Ajuste com as linhas planejadas por classe (notebook 04m)", conta(y_aj) == N_AJ, f"({len(X_aj):,} linhas)")
verifica("Validação com as linhas planejadas por classe (notebook 04m)", conta(y_va) == N_VA, f"({len(X_va):,} linhas)")
verifica("Refit = ajuste + validação: todas as linhas selecionadas, por classe", conta(y_rf) == {c: N_AJ[c] + N_VA[c] for c in CLASSES} and set(aj["id_linha"]) | set(va["id_linha"]) == set(rf_df["id_linha"]) and not set(aj["id_linha"]) & set(va["id_linha"]), f"({len(X_rf):,} linhas)")
verifica("Refit: a view e a tabela de seleção têm as mesmas linhas", len(rf_df) == n_linhas(TAB_REFIT) == len(refit_sel))
verifica("Pesos de classe (balanced) conferem com o scikit-learn", all(abs(PESO_CLASSE_AJ[CLASSES[k]] - w) < 1e-12 for k, w in enumerate(compute_class_weight("balanced", classes=np.arange(K), y=y_aj))))
verifica("Probabilidades finitas, entre 0 e 1, e cada linha soma 1", bool(np.isfinite(proba_va).all() and ((proba_va >= 0) & (proba_va <= 1)).all() and np.allclose(proba_va.sum(axis=1), 1.0)))
verifica("Matrizes de confusão fecham: soma = linhas da validação (e = soma dos pesos, ponderada)", int(arr_sem["cm"].sum()) == len(y_va) and abs(arr_prop["cm"].sum() - float(w_va.sum())) < 1e-6)
verifica("Pesos da validação somam as linhas de cada classe no treino", all(abs(float(w_va[y_va == COD[c]].sum()) - N_TREINO[c]) < 1e-6 * N_TREINO[c] for c in CLASSES))
verifica("Métricas que não dependem da proporção são iguais nas duas colunas", abs(m_sem["acuracia_balanceada"] - m_prop["acuracia_balanceada"]) < 1e-9 and np.allclose(arr_sem["recall"], arr_prop["recall"], atol=1e-12))
verifica("Arquivos esperados gravados", all((PASTA / f).exists() for f in ("modelo_ajuste.joblib", "modelo_final_refit.joblib", "config_treino.json", "preprocessamento.json", "classes.json", "metricas_validacao.json", "metricas_validacao.csv",
                                                                          "metricas_por_classe_validacao.csv", "matriz_confusao_validacao.csv", "dispersao_particoes_validacao.csv", "cv_ajuste.csv", "cv_recall_por_classe.csv", "custo_computacional.json", "custo_computacional.csv", "refit_ids.csv")))
verifica("Custo medido: tempos positivos" + ("" if psutil else " (memória não medida: psutil ausente)"), fit_ajuste["relogio_s"] > 0 and fit_refit["relogio_mediana_s"] > 0 and inf_fin[0]["vazao_fim_a_fim_fluxos_s"] > 0)
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print(f"\nPróximo passo: notebook 06m (avaliação no teste), apontando para {PASTA.name}")

✔ Modelo do ajuste gravado reproduz as probabilidades da validação 
✔ Modelo final gravado reproduz as previsões do modelo em memória 
✔ Ordem dos atributos gravada = ordem usada no treino e dentro do modelo 
✔ Medianas gravadas = medianas da tabela de auditoria do notebook 03m 
✔ Hiperparâmetros do modelo = configuração escolhida (inclui class_weight='balanced') 
✔ O modelo aprendeu as classes 0..K-1, na ordem de classes.json (7 classes)
✔ Ajuste com as linhas planejadas por classe (notebook 04m) (239,671 linhas)
✔ Validação com as linhas planejadas por classe (notebook 04m) (59,918 linhas)
✔ Refit = ajuste + validação: todas as linhas selecionadas, por classe (299,589 linhas)
  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
✔ Refit: a view e a tabela de seleção têm as mesmas linhas 
✔ Pesos de classe (balanced) conferem com o scikit-learn 
✔ Probabilidades finitas, entre 0 e 1, e cada linha soma 1 
✔ Matrizes de confusão 

In [14]:
cols_fin = set(consultar("SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = 'prep_final_treino'")["name"])
col_conf = next((c for c in ("conflito_rotulo", "conflito_binario") if c in cols_fin), None)
assert col_conf, "prep_final_treino não tem coluna de conflito de rótulo"
conf = consultar(f"SELECT toString(id_linha) AS id_linha, {col_conf} AS conf FROM prep_final_treino WHERE id_linha IN (SELECT id_linha FROM {SEL_TAB} WHERE papel = 'validacao')")
d = pd.DataFrame({"id_linha": va["id_linha"].astype(str).values, "real": y_va, "prev": proba_va.argmax(axis=1)}).merge(conf.assign(conf=pd.to_numeric(conf["conf"])), on="id_linha", how="left")
assert len(d) == len(va) and d["conf"].notna().all(), "nem todas as linhas da validação foram encontradas"
d["acerto"] = (d.real == d.prev); d["classe"] = d.real.map(lambda k: CLASSES[k]); d["conf"] = d["conf"].astype(int)
t = d.groupby(["classe", "conf"]).agg(linhas=("acerto", "size"), recall=("acerto", "mean")).unstack("conf")
t.columns = [f"{a} ({'com marca' if b == 1 else 'sem marca'})" for a, b in t.columns]; t = t.reindex(CLASSES)
for c_ in [c for c in t.columns if c.startswith('linhas')]: t[c_] = t[c_].fillna(0).astype(int)
print(f"Marca usada: {col_conf} (calculada com as 13 classes do treino)\n"); display(t.round(4))
tot = d.groupby("conf").agg(linhas=("acerto", "size"), acuracia=("acerto", "mean"), erros=("acerto", lambda s: int((~s).sum())))
print("\nGeral:"); display(tot.round(4))
print(f"Dos {int((~d.acerto).sum()):,} erros da validação, {int(((~d.acerto) & (d.conf == 1)).sum()):,} ({100 * ((~d.acerto) & (d.conf == 1)).sum() / max((~d.acerto).sum(), 1):.1f}%) estão em linhas com marca de conflito; essas linhas são {100 * (d.conf == 1).mean():.1f}% da validação.")

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
Marca usada: conflito_rotulo (calculada com as 13 classes do treino)



,linhas (sem marca),linhas (com marca),recall (sem marca),recall (com marca)
classe,,,,
BENIGN,9996,4,0.9993,0.2500
LDAP,3431,2589,0.9816,0.9421
MSSQL,9012,988,0.9399,0.3381
NetBIOS,1899,1999,0.9126,0.6208
Syn,9822,178,0.9671,0.2865
UDP,8627,1373,0.9378,0.7939
UDPLag,8785,1215,0.8785,0.5440



Geral:


,linhas,acuracia,erros
conf,,,
0,51572,0.9475,2705
1,8346,0.6970,2529


Dos 5,234 erros da validação, 2,529 (48.3%) estão em linhas com marca de conflito; essas linhas são 13.9% da validação.
